# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² Mass Spectrometry EV dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using a Croissant schema and accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get metadata as a Python object
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

We enumerate all available RecordSets, their fields, and corresponding field and column `@id`s using the `metadata.record_sets` attribute.

In [ ]:
# List RecordSets and their Fields/Columns by @id
record_set_infos = []
print('Available RecordSets in the dataset:')
for rs in metadata.record_sets:
    print(f"- RecordSet name: {getattr(rs, 'name', None)} | @id: {getattr(rs, '@id', None)}")
    field_ids = []
    if hasattr(rs, 'fields') and rs.fields:
        for fld in rs.fields:
            print(f"    Field: {getattr(fld, 'name', None)} | @id: {getattr(fld, '@id', None)} | type: {getattr(fld, 'data_type', None)}")
            field_ids.append(getattr(fld, '@id', None))
            # If field has a column, display column info
            if hasattr(fld, 'columns') and fld.columns:
                for col in fld.columns:
                    print(f"        Column: {getattr(col, 'name', None)} | @id: {getattr(col, '@id', None)}")
    record_set_infos.append({'@id': getattr(rs, '@id', None), 'name': getattr(rs, 'name', None), 'field_ids': field_ids})

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for further processing. Use the record set and field `@id` values identified in the previous step.

<sup>Note: For this dataset there may be multiple RecordSets; you can select the primary one for most analysis.</sup>

In [ ]:
# Define all record sets by their @id
record_set_ids = [rs['@id'] for rs in record_set_infos if rs['@id'] is not None]

dataframes = {}
print(f"Loading records of RecordSets: {record_set_ids}")
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet {record_set_id} - Loaded {df.shape[0]} rows, {df.shape[1]} columns.")
        print(f"    Columns: {df.columns.tolist()}")
    else:
        print(f"RecordSet {record_set_id} returned zero records.")
# List the first few rows of the main data table (choose the largest or most relevant RecordSet)
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nPreview of primary RecordSet ({main_record_set_id}):")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll process and examine one numeric field—such as protein abundance, coverage, or molecular weight—using its field `@id` (and column label).

The workflow includes:
1. Filtering the DataFrame for a chosen numeric field exceeding a threshold.
2. Normalizing this field.
3. Optionally, grouping by another field like 'description' or a category.

Replace `<numeric_field_id>` and `<group_field_id>` with actual `@id`s from the previous overview, and adjust the column names as used in your DataFrame.

In [ ]:
# Example --- update these with valid values from your dataset!
# See record_set_infos content above to find desired field and group @id/name mapping.

primary_record_set_id = main_record_set_id  # Use the largest/relevant
df = dataframes[primary_record_set_id]

# Pick a numeric field and a group field by inspecting df.columns
# Example field names below; replace with your dataset's real column names
numeric_field = None
group_field = None
# Try to automatically detect likely numeric and group fields
for col in df.columns:
    if (('abundance' in col.lower()) or ('coverage' in col.lower()) or ('mw' in col.lower())) and numeric_field is None:
        # Prefer abundance/coverage/mw for demonstration
        numeric_field = col
    elif (('type' in col.lower()) or ('sample' in col.lower()) or ('description' in col.lower())) and group_field is None:
        group_field = col

if numeric_field is None:
    numeric_field = df.select_dtypes('number').columns[0]  # fallback

print(f"Selected numeric field: {numeric_field}")
if group_field:
    print(f"Selected group/categorical field: {group_field}")

# 1. Filter records with numeric_field > threshold (choose threshold based on data)
if numeric_field and (df[numeric_field].dtype in [float, int] or pd.api.types.is_numeric_dtype(df[numeric_field])):
    threshold = float(df[numeric_field].quantile(0.8))  # 80th percentile as threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered (>{threshold:.2f}) records for '{numeric_field}': {len(filtered_df)}/{len(df)}")
    display(filtered_df.head())

    # 2. Normalize numeric_field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}':")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # 3. Group by group_field (if exists)
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean '{numeric_field}' by '{group_field}':")
        display(grouped_df.head())
else:
    print(f"No numeric field detected: columns are {df.columns.tolist()}")

## 5. Visualization
Visualize data distributions and field relationships with matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field selected for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to use `mlcroissant` to programmatically examine a multi-table mass spectrometry protein dataset described with a FAIR Croissant schema:

- Loaded Croissant metadata and identified record set and field structure via their `@id` values
- Loaded main data tables and explored records and fields
- Performed basic cleaning, filtering, normalization, and grouping
- Created quick visual summaries

This workflow can be extended to more advanced analyses, ML preparation, and data integrations using the precise provenance and schema support of Croissant/`mlcroissant`.

**For more details on this dataset, visit the [Frontiers entry](https://sen.science/doi/10.71728/senscience.vq4a-28xa).**